# Decision Tree — Kotak vs Segitiga (Supervised)

Notebook ini membuat dataset sintetis 2D: **kelas kotak** vs **kelas segitiga**, lalu melakukan **klasifikasi supervised** menggunakan Decision Tree.

Intinya: kalau ada label `y` (0=kotak, 1=segitiga), maka ini **supervised** (klasifikasi).

In [ ]:
# (Opsional) install dependency jika kernel belum punya
%pip -q install scikit-learn matplotlib seaborn numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid')
rng = np.random.default_rng(42)

In [ ]:
def sample_square(n, rng, xlim=(0.0, 1.0), ylim=(0.0, 1.0)):
    """Sample titik uniform di dalam kotak axis-aligned."""
    x = rng.uniform(xlim[0], xlim[1], size=n)
    y = rng.uniform(ylim[0], ylim[1], size=n)
    return np.c_[x, y]


def sample_right_triangle(n, rng, a=(0.0, 0.0), b=(1.0, 0.0), c=(0.0, 1.0)):
    """
    Sample titik uniform di dalam segitiga (a,b,c) menggunakan barycentric coords.
    Default: segitiga siku-siku dengan vertices (0,0), (1,0), (0,1).
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)

    u = rng.uniform(0, 1, size=n)
    v = rng.uniform(0, 1, size=n)
    swap = (u + v) > 1
    u[swap] = 1 - u[swap]
    v[swap] = 1 - v[swap]

    return a + u[:, None] * (b - a) + v[:, None] * (c - a)

In [ ]:
# Buat dataset
n_per_class = 500
noise_std = 0.04  # naikkan jika ingin lebih 'berisik'

X0 = sample_square(n_per_class, rng, xlim=(-0.2, 1.2), ylim=(-0.2, 1.2))
X1 = sample_right_triangle(n_per_class, rng, a=(0.0, 0.0), b=(1.4, 0.1), c=(0.1, 1.4))

# Tambah sedikit noise (opsional)
X0 = X0 + rng.normal(0, noise_std, size=X0.shape)
X1 = X1 + rng.normal(0, noise_std, size=X1.shape)

X = np.vstack([X0, X1])
y = np.concatenate([np.zeros(n_per_class, dtype=int), np.ones(n_per_class, dtype=int)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu', s=16, alpha=0.8, edgecolor='none')
plt.title('Dataset: kotak (0) vs segitiga (1) — TRAIN')
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

In [ ]:
def plot_decision_boundary(model, X, y, title=None, grid_step=0.02):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, grid_step),
        np.arange(y_min, y_max, grid_step),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    if hasattr(model, 'predict_proba'):
        zz = model.predict_proba(grid)[:, 1]
    else:
        zz = model.predict(grid)
    zz = zz.reshape(xx.shape)

    plt.contourf(xx, yy, zz, levels=30, cmap='RdBu', alpha=0.35)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', s=18, edgecolor='none')
    plt.xlabel('x1')
    plt.ylabel('x2')
    if title:
        plt.title(title)

In [ ]:
# Model klasifikasi: Decision Tree
# Coba ubah max_depth untuk melihat underfit/overfit
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'SQUARE vs TRIANGLE | depth=5 | train={train_acc:.3f} | test={test_acc:.3f}')

plt.figure(figsize=(6, 4))
plot_decision_boundary(
    clf, X_test, y_test,
    title=f'Decision boundary (depth=5)\ntrain={train_acc:.3f} | test={test_acc:.3f}'
)
plt.show()

## Catatan
- Ini **supervised** karena ada label `y` (0=kotak, 1=segitiga).
- Kalau label dihilangkan (tanpa `y`), barulah kita masuk ke **unsupervised** (mis. clustering dengan KMeans), tapi hasil cluster belum tentu persis sesuai bentuk.
- Untuk melihat underfit/overfit, coba ubah `max_depth` (mis. 1, 3, 10, None).